In [19]:
import pandas as pd

df = pd.read_csv('a_hostel_static.csv')

In [20]:
df.head()

,id,time,amplitude1_1,amplitude2_1,amplitude3_1,amplitude4_1,amplitude5_1,amplitude6_1,amplitude7_1,amplitude8_1,...,phase58_2,phase59_2,phase60_2,phase61_2,phase62_2,phase63_2,phase64_2,rssi_1,rssi_2,activity
0,0,23:19:49,115.277925,4.0,42.485292,39.824616,36.359318,31.906112,25.553865,21.023796,...,-2.221873,-2.205823,-2.158799,-2.057696,-1.910089,-1.681454,-1.570796,-52,-51,0
1,1,23:19:49,115.277925,4.0,41.231056,38.483763,34.438351,30.413813,24.596748,19.313208,...,2.251318,2.270689,2.293776,2.427502,2.582993,2.788602,2.953371,-52,-51,0
2,2,23:19:49,115.277925,4.0,42.426407,38.948684,35.468296,30.479501,25.495098,20.518285,...,-0.110657,-0.119429,-0.090660,0.000000,0.124355,0.463648,0.702257,-52,-52,0
3,3,23:19:49,115.277925,4.0,43.324358,39.560081,36.769553,31.780497,26.000000,21.189620,...,1.387686,1.343997,1.373401,1.365401,1.441094,1.570796,1.657535,-52,-52,0
4,4,23:19:49,115.277925,4.0,38.832976,36.400549,32.649655,29.068884,23.430749,18.601075,...,-1.208252,-1.227772,-1.216091,-1.203622,-1.156289,-1.091277,-1.030377,-52,-51,0


In [21]:
import joblib

model = joblib.load('./models/gmm_model_0.pkl')

In [22]:
import joblib

pca = joblib.load('./models/pca_model.joblib')

In [23]:
import pywt
import numpy as np

wavelet_name = 'gaus1'

scales = np.arange(1, 128)	

In [24]:
import numpy as np

def getWavelet(data):
	X = data.drop(columns = ['activity', 'id', 'time'])
	pca_X = pd.DataFrame(pca.transform(X))
	coefficients, frequencies = pywt.cwt(pca_X.iloc[: , 0], scales, wavelet_name)
	wavelet_variances = np.sum(coefficients**2, axis=0)
	return wavelet_variances



In [25]:
def get25(group):
	fraction = len(group) /25
	forPredict = []
	for i in range(1,26):
		index = int(i * fraction)
		forPredict.append(group.iloc[index-1])

	return forPredict

In [26]:
grouped_data = df.groupby('time')
i = 0
wavelet_list = np.zeros((1, 25))

for time, group in grouped_data:
	if len(group) > 25:
		forPredict = get25(group)
		wavelet = getWavelet(pd.DataFrame(forPredict))
		wavelet_list = np.vstack([wavelet_list, wavelet])

wavelet_list = np.delete(wavelet_list, np.s_[:1], axis=0)

wavelet_list = pd.DataFrame(wavelet_list)

wavelet_list.columns = [str(i) for i in range(0, 25)]

model.fit(wavelet_list)



GaussianMixture(n_components=3)

In [27]:
joblib.dump(model, './models/gmm_model_0.pkl')

['./models/gmm_model_0.pkl']